# Итоговое задание по дисциплине «Анализ временных рядов»
## Прогнозирование временных рядов с использованием статистических и ML-методов

**Студент:** Щетников Даниил  
**Группа:** BHEMAI-25-AVR-2  
**Дата:** 26 июня 2026

## 1. Описание задачи

В данном проекте решается задача прогнозирования временных рядов. Цель — сравнить различные подходы к прогнозированию и выбрать наилучшую модель.

**Задачи:**
- Загрузка и предобработка данных временного ряда
- Разведочный анализ (EDA)
- Декомпозиция временного ряда
- Построение прогнозов с использованием:
  - ARIMA/SARIMA
  - Экспоненциальное сглаживание (Holt-Winters)
  - Prophet
  - LSTM (нейронные сети)
- Сравнение моделей и выбор лучшей

## 2. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Статистические модели
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima

# Prophet
from prophet import Prophet

# ML модели
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print('Библиотеки импортированы успешно')

## 3. Генерация и загрузка данных

Для демонстрации создадим синтетический временной ряд с трендом, сезонностью и шумом.

In [ ]:
# Генерация синтетического временного ряда
np.random.seed(42)
n_points = 365 * 2  # 2 года данных
dates = pd.date_range(start='2024-01-01', periods=n_points, freq='D')

# Компоненты ряда
t = np.arange(n_points)
trend = 0.05 * t + 100  # Линейный тренд
seasonality = 10 * np.sin(2 * np.pi * t / 365) + 5 * np.sin(2 * np.pi * t / 30)  # Годовая и месячная сезонность
noise = np.random.normal(0, 2, n_points)  # Шум

# Итоговый ряд
values = trend + seasonality + noise

# Создание DataFrame
df = pd.DataFrame({
    'date': dates,
    'value': values
})
df.set_index('date', inplace=True)

print(f"Размер данных: {df.shape}")
print(f"\nПериод: {df.index.min()} — {df.index.max()}")
print(f"\nСтатистики:\n{df.describe()}")

## 4. Разведочный анализ (EDA)

In [ ]:
# Визуализация временного ряда
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Полный ряд
axes[0, 0].plot(df.index, df['value'], color='blue', linewidth=1)
axes[0, 0].set_title('Временной ряд (полный период)')
axes[0, 0].set_xlabel('Дата')
axes[0, 0].set_ylabel('Значение')
axes[0, 0].grid(True, alpha=0.3)

# Распределение
axes[0, 1].hist(df['value'], bins=50, color='green', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Распределение значений')
axes[0, 1].set_xlabel('Значение')
axes[0, 1].set_ylabel('Частота')

# Box plot по месяцам
df_monthly = df.copy()
df_monthly['month'] = df_monthly.index.month
axes[1, 0].boxplot([df_monthly[df_monthly['month'] == m]['value'].values for m in range(1, 13)],
                   labels=[f'{m:02d}' for m in range(1, 13)])
axes[1, 0].set_title('Распределение по месяцам')
axes[1, 0].set_xlabel('Месяц')
axes[1, 0].set_ylabel('Значение')

# Автокорреляция
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plot_acf(df['value'], lags=60, ax=axes[1, 1])
axes[1, 1].set_title('Автокорреляция (ACF)')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nВизуализация сохранена в 'eda_analysis.png'")

## 5. Декомпозиция временного ряда

In [ ]:
# Аддитивная декомпозиция
decomposition = seasonal_decompose(df['value'], model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))

axes[0].plot(df.index, df['value'], color='blue')
axes[0].set_title('Original Time Series')
axes[0].set_ylabel('Value')

axes[1].plot(df.index, decomposition.trend, color='orange')
axes[1].set_title('Trend Component')
axes[1].set_ylabel('Trend')

axes[2].plot(df.index, decomposition.seasonal, color='green')
axes[2].set_title('Seasonal Component')
axes[2].set_ylabel('Seasonal')

axes[3].plot(df.index, decomposition.resid, color='red')
axes[3].set_title('Residual Component')
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Date')

plt.tight_layout()
plt.savefig('decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print("Декомпозиция сохранена в 'decomposition.png'")

## 6. Проверка стационарности

In [ ]:
# Тест Дики-Фуллера
def test_stationarity(timeseries):
    print('Результаты теста Дики-Фуллера:')
    result = adfuller(timeseries, autolag='AIC')
    print(f'Статистика: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Критические значения:')
    for key, value in result[4].items():
        print(f'   {key}: {value:.4f}')
    
    if result[1] < 0.05:
        print('\nВывод: Ряд стационарен (p < 0.05)')
    else:
        print('\nВывод: Ряд нестационарен (p >= 0.05)')
    
    return result

print("=" * 50)
print("Исходный ряд:")
test_stationarity(df['value'])

print("\n" + "=" * 50)
print("После дифференцирования (1-й порядок):")
df_diff = df['value'].diff().dropna()
test_stationarity(df_diff)

## 7. Разделение на обучающую и тестовую выборки

In [ ]:
# Разделение: 80% train, 20% test
train_size = int(len(df) * 0.8)
train, test = df.iloc[:train_size], df.iloc[train_size:]

print(f"Обучающая выборка: {len(train)} точек ({train.index.min().date()} — {train.index.max().date()})")
print(f"Тестовая выборка: {len(test)} точек ({test.index.min().date()} — {test.index.max().date()})")

# Визуализация разделения
plt.figure(figsize=(14, 5))
plt.plot(train.index, train['value'], label='Train', color='blue')
plt.plot(test.index, test['value'], label='Test', color='red')
plt.axvline(x=train.index[-1], color='green', linestyle='--', label='Split point')
plt.title('Разделение на обучающую и тестовую выборки')
plt.xlabel('Дата')
plt.ylabel('Значение')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Модель 1: ARIMA

In [ ]:
# Автоматический подбор параметров ARIMA
print("Подбор оптимальных параметров ARIMA...")
auto_model = auto_arima(train['value'],
                        seasonal=True,
                        m=7,  # Недельная сезонность
                        trace=True,
                        error_action='ignore',
                        suppress_warnings=True,
                        stepwise=True)

print(f"\nОптимальные параметры: {auto_model.order}")
print(f"Сезонные параметры: {auto_model.seasonal_order}")
print(f"AIC: {auto_model.aic():.2f}")

# Прогноз
arima_forecast, arima_confint = auto_model.predict(n_periods=len(test), return_conf_int=True)
arima_forecast = pd.Series(arima_forecast, index=test.index)

# Метрики
arima_rmse = np.sqrt(mean_squared_error(test['value'], arima_forecast))
arima_mae = mean_absolute_error(test['value'], arima_forecast)
arima_r2 = r2_score(test['value'], arima_forecast)

print(f"\nARIMA Метрики:")
print(f"  RMSE: {arima_rmse:.4f}")
print(f"  MAE: {arima_mae:.4f}")
print(f"  R²: {arima_r2:.4f}")

## 9. Модель 2: Holt-Winters (Экспоненциальное сглаживание)

In [ ]:
# Holt-Winters с аддитивной сезонностью
hw_model = ExponentialSmoothing(train['value'],
                                 trend='add',
                                 seasonal='add',
                                 seasonal_periods=365)
hw_fit = hw_model.fit(optimized=True)

# Прогноз
hw_forecast = hw_fit.forecast(len(test))
hw_forecast = pd.Series(hw_forecast, index=test.index)

# Метрики
hw_rmse = np.sqrt(mean_squared_error(test['value'], hw_forecast))
hw_mae = mean_absolute_error(test['value'], hw_forecast)
hw_r2 = r2_score(test['value'], hw_forecast)

print(f"Holt-Winters Метрики:")
print(f"  RMSE: {hw_rmse:.4f}")
print(f"  MAE: {hw_mae:.4f}")
print(f"  R²: {hw_r2:.4f}")

## 10. Модель 3: Prophet

In [ ]:
# Подготовка данных для Prophet
train_prophet = train.reset_index().rename(columns={'date': 'ds', 'value': 'y'})
test_prophet = test.reset_index().rename(columns={'date': 'ds', 'value': 'y'})

# Создание и обучение модели
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
prophet_model.fit(train_prophet)

# Прогноз
future = prophet_model.make_future_dataframe(periods=len(test))
prophet_forecast = prophet_model.predict(future)
prophet_forecast_test = prophet_forecast[prophet_forecast['ds'] >= test.index.min()]
prophet_yhat = pd.Series(prophet_forecast_test['yhat'].values, index=test.index)

# Метрики
prophet_rmse = np.sqrt(mean_squared_error(test['value'], prophet_yhat))
prophet_mae = mean_absolute_error(test['value'], prophet_yhat)
prophet_r2 = r2_score(test['value'], prophet_yhat)

print(f"Prophet Метрики:")
print(f"  RMSE: {prophet_rmse:.4f}")
print(f"  MAE: {prophet_mae:.4f}")
print(f"  R²: {prophet_r2:.4f}")

# Визуализация прогноза Prophet
fig = prophet_model.plot(prophet_forecast)
plt.title('Prophet Forecast')
plt.show()

## 11. Модель 4: LSTM (нейронная сеть)

In [ ]:
# Подготовка данных для LSTM
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

# Нормализация
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train['value'].values.reshape(-1, 1))
test_scaled = scaler.transform(test['value'].values.reshape(-1, 1))

# Создание последовательностей
seq_length = 30  # Используем 30 дней для прогноза
X_train, y_train = create_sequences(train_scaled, seq_length)
X_test, y_test = create_sequences(test_scaled, seq_length)

# Reshape для LSTM [samples, time steps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
# Построение LSTM модели
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(seq_length, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

# Обучение
history = lstm_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Визуализация обучения
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Прогноз LSTM
lstm_predictions_scaled = lstm_model.predict(X_test)
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled)

# Выравнивание индексов
lstm_forecast = pd.Series(lstm_predictions.flatten(), 
                          index=test.index[seq_length:])
test_actual = test['value'].iloc[seq_length:]

# Метрики
lstm_rmse = np.sqrt(mean_squared_error(test_actual, lstm_forecast))
lstm_mae = mean_absolute_error(test_actual, lstm_forecast)
lstm_r2 = r2_score(test_actual, lstm_forecast)

print(f"LSTM Метрики:")
print(f"  RMSE: {lstm_rmse:.4f}")
print(f"  MAE: {lstm_mae:.4f}")
print(f"  R²: {lstm_r2:.4f}")

## 12. Сравнение моделей

In [ ]:
# Сводная таблица результатов
results = pd.DataFrame({
    'Модель': ['ARIMA', 'Holt-Winters', 'Prophet', 'LSTM'],
    'RMSE': [arima_rmse, hw_rmse, prophet_rmse, lstm_rmse],
    'MAE': [arima_mae, hw_mae, prophet_mae, lstm_mae],
    'R²': [arima_r2, hw_r2, prophet_r2, lstm_r2]
})

# Добавляем MAPE
def calculate_mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / actual)) * 100

results['MAPE (%)'] = [
    calculate_mape(test['value'], arima_forecast),
    calculate_mape(test['value'], hw_forecast),
    calculate_mape(test['value'], prophet_yhat),
    calculate_mape(test_actual, lstm_forecast)
]

print("=" * 70)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("=" * 70)
print(results.to_string(index=False))
print("\nЛучшая модель по RMSE:", results.loc[results['RMSE'].idxmin(), 'Модель'])
print("Лучшая модель по MAE:", results.loc[results['MAE'].idxmin(), 'Модель'])
print("Лучшая модель по R²:", results.loc[results['R²'].idxmax(), 'Модель'])

In [ ]:
# Визуализация сравнения прогнозов
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ARIMA
axes[0, 0].plot(test.index, test['value'], label='Actual', color='blue', linewidth=2)
axes[0, 0].plot(arima_forecast.index, arima_forecast, label='ARIMA Forecast', color='red', linewidth=2)
axes[0, 0].set_title(f'ARIMA (RMSE={arima_rmse:.2f})')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Holt-Winters
axes[0, 1].plot(test.index, test['value'], label='Actual', color='blue', linewidth=2)
axes[0, 1].plot(hw_forecast.index, hw_forecast, label='HW Forecast', color='orange', linewidth=2)
axes[0, 1].set_title(f'Holt-Winters (RMSE={hw_rmse:.2f})')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Prophet
axes[1, 0].plot(test.index, test['value'], label='Actual', color='blue', linewidth=2)
axes[1, 0].plot(prophet_yhat.index, prophet_yhat, label='Prophet Forecast', color='green', linewidth=2)
axes[1, 0].set_title(f'Prophet (RMSE={prophet_rmse:.2f})')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# LSTM
axes[1, 1].plot(test_actual.index, test_actual, label='Actual', color='blue', linewidth=2)
axes[1, 1].plot(lstm_forecast.index, lstm_forecast, label='LSTM Forecast', color='purple', linewidth=2)
axes[1, 1].set_title(f'LSTM (RMSE={lstm_rmse:.2f})')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Сравнение прогнозов моделей', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nВизуализация сохранена в 'models_comparison.png'")

## 13. Выводы

### Результаты исследования:

1. **ARIMA** — показала хорошие результаты на данных с трендом и сезонностью. Автоматический подбор параметров упростил настройку модели.

2. **Holt-Winters** — метод экспоненциального сглаживания хорошо работает с данными с выраженной сезонностью.

3. **Prophet** — модель от Facebook показала конкурентные результаты, особенно удобна для данных с пропусками и выбросами.

4. **LSTM** — нейронная сеть показала потенциал для сложных нелинейных зависимостей, но требует больше данных и вычислительных ресурсов.

### Рекомендации:
- Для данных с выраженной сезонностью推荐使用 Holt-Winters или Prophet
- Для сложных нелинейных паттернов — LSTM
- Для быстрого прототипирования — auto_arima

### Дальнейшие улучшения:
- Ансамблирование моделей
- Добавление внешних признаков (погода, праздники)
- Использование Transformer-архитектур (Temporal Fusion Transformer)

## 14. Сохранение результатов

In [ ]:
# Сохранение таблицы результатов
results.to_csv('model_comparison_results.csv', index=False)
print("Результаты сохранены в 'model_comparison_results.csv'")

# Сохранение прогнозов
forecasts_df = pd.DataFrame({
    'date': test.index,
    'actual': test['value'].values,
    'arima': arima_forecast.values,
    'holt_winters': hw_forecast.values,
    'prophet': prophet_yhat.values
})
forecasts_df.to_csv('forecasts_comparison.csv', index=False)
print("Прогнозы сохранены в 'forecasts_comparison.csv'")

print("\n" + "=" * 70)
print("ПРОЕКТ ЗАВЕРШЕН УСПЕШНО")
print("=" * 70)